# Data Collection: Grok (xAI API)

This notebook queries Grok via the xAI API for each question in the question bank and saves responses to `data/responses/grok.jsonl`.

The xAI API is OpenAI-compatible, so the `openai` Python package is used with a custom base URL.

**Before running:** complete the setup in [Environment Setup](setup.md) and ensure `XAI_API_KEY` is set.

In [ ]:
import openai
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

In [ ]:
MODEL = "grok-2"
TEMPERATURE = 0.0
MAX_TOKENS = 1024
OUTPUT_FILE = Path("data/responses/grok.jsonl")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# xAI API is OpenAI-compatible
client = openai.OpenAI(
    api_key=os.environ["XAI_API_KEY"],
    base_url="https://api.x.ai/v1",
)

SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about early church history.
Provide accurate, detailed, and nuanced responses based on historical scholarship.
Acknowledge uncertainty or scholarly debate where it exists.
Do not refuse historically answerable questions on the grounds of contemporary theological controversy."""

questions = pd.read_csv("../figures/question_bank.csv")
print(f"Loaded {len(questions)} questions")

In [ ]:
with open(OUTPUT_FILE, "w") as f:
    for _, row in questions.iterrows():
        print(f"Querying {row['question_id']}...", end=" ")
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": row["question"]},
                ],
            )
            record = {
                "question_id": row["question_id"],
                "figure": row["figure"],
                "model": MODEL,
                "model_version": response.model,
                "prompt": row["question"],
                "response": response.choices[0].message.content,
                "temperature": TEMPERATURE,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "tokens_used": response.usage.total_tokens,
            }
            f.write(json.dumps(record) + "\n")
            print(f"done ({record['tokens_used']} tokens)")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(0.5)

print(f"\nDone. Responses saved to {OUTPUT_FILE}")